In [1]:
from pathlib import Path
import pandas as pd

base = Path("../data/raw")
files = sorted(base.glob("train-*-of-00171.parquet"))
print(files)

if not files:
    raise FileNotFoundError(f"Parquet files not found in {base.resolve()}")

dfs = [pd.read_parquet(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

[WindowsPath('../data/raw/train-00000-of-00171.parquet'), WindowsPath('../data/raw/train-00001-of-00171.parquet')]


In [2]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   audio                 1230 non-null   object        
 1   title                 1230 non-null   str           
 2   url                   1230 non-null   str           
 3   artist                1230 non-null   str           
 4   composer              1230 non-null   str           
 5   lyricist              1230 non-null   str           
 6   publisher             1230 non-null   str           
 7   genres                1230 non-null   object        
 8   tags                  1230 non-null   object        
 9   released              1230 non-null   datetime64[ms]
 10  language              1230 non-null   str           
 11  listens               1230 non-null   uint64        
 12  artist_url            1230 non-null   str           
 13  artist_website        1230 no

,audio,title,url,artist,composer,lyricist,publisher,genres,tags,released,...,album_title,album_url,license,copyright,explicit,instrumental,allow_commercial_use,allow_derivatives,require_attribution,require_share_alike
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...,Food,http://freemusicarchive.org/music/AWOL/AWOL_-_...,AWOL,,,,[70],[],2008-11-26 01:48:12,...,AWOL - A Way Of Life,http://freemusicarchive.org/music/AWOL/AWOL_-_...,18,,1.0,0,0,1,1,1
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TIT2\x...,Electric Ave,http://freemusicarchive.org/music/AWOL/AWOL_-_...,AWOL,,,,[70],[],2008-11-26 01:48:14,...,AWOL - A Way Of Life,http://freemusicarchive.org/music/AWOL/AWOL_-_...,18,,1.0,0,0,1,1,1
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,This World,http://freemusicarchive.org/music/AWOL/AWOL_-_...,AWOL,,,,[70],[],2008-11-26 01:48:20,...,AWOL - A Way Of Life,http://freemusicarchive.org/music/AWOL/AWOL_-_...,18,,1.0,0,0,1,1,1
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...,Freeway,http://freemusicarchive.org/music/Kurt_Vile/Co...,Kurt Vile,Kurt Vile,,,[116],[],2008-11-25 17:49:06,...,Constant Hitmaker,http://freemusicarchive.org/music/Kurt_Vile/Co...,13,,0.0,0,0,0,1,0
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05YTIT2\x...,Spiritual Level,http://freemusicarchive.org/music/Chris_and_Ni...,Nicky Cook,,,,"[54, 136]",[],2008-11-26 01:48:56,...,Niris,http://freemusicarchive.org/music/Chris_and_Ni...,13,,NaN,0,0,0,1,0


In [3]:
import numpy as np

all_genres = set()

for g in df["genres"]:

    if isinstance(g, np.ndarray):
        g_iter = g.tolist()
    elif isinstance(g, (list, tuple, set)):
        g_iter = list(g)

    for item in g_iter:
        all_genres.add(str(item))

print("num genres:", len(all_genres))
print("sample:", list(all_genres)[:50])

num genres: 47
sample: ['49', '149', '92', '97', '2', '130', '75', '80', '125', '42', '121', '129', '6', '122', '44', '9', '119', '82', '144', '94', '70', '91', '88', '90', '116', '10', '123', '56', '50', '118', '136', '79', '60', '107', '117', '54', '77', '58', '96', '111', '61', '51', '18', '5', '106', '53', '84']


In [4]:
cols = ["audio", "title", "artist"]
available = [c for c in cols if c in df.columns]
print("available cols:", available)

df_small = df[available].copy()
df_small.info()
df_small.head()

available cols: ['audio', 'title', 'artist']
<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   audio   1230 non-null   object
 1   title   1230 non-null   str   
 2   artist  1230 non-null   str   
dtypes: object(1), str(2)
memory usage: 62.2+ KB


,audio,title,artist
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...,Food,AWOL
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TIT2\x...,Electric Ave,AWOL
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,This World,AWOL
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...,Freeway,Kurt Vile
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05YTIT2\x...,Spiritual Level,Nicky Cook


In [5]:
print(df["artist"].nunique())

160


In [7]:
out_dir = Path("../data/processed")

output_path = out_dir / "data.parquet"
df_small.to_parquet(output_path, index=False)


In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet("data/processed/data.parquet")

def build_prompt(title, artist):
    return f"""
Ты — музыкальный критик и рекомендательная система.
На вход я даю тебе один аудиотрек и его метаданные: название и исполнитель.
Твоя задача — кратко описать музыку (не текст песни) простым человеческим языком для рекомендательной системы.

Правила:
- Описание 1–2 предложения, максимум 40–50 слов.
- Упоминай жанр и поджанр.
- Опиши настроение (спокойное, меланхоличное, энергичное и т.п.).
- Укажи основные инструменты/звучание.
- Оцени темп (медленный/средний/быстрый) и уровень энергии.
- Не пиши название песни и имя исполнителя в описании.
- Не используй общие фразы, пиши конкретно и по делу.

Метаданные трека:
- Название: "{title}"
- Исполнитель: "{artist}"

Сформируй только описание, без пояснений и списков.
""".strip()

def describe_track(audio_path, artist, title):
    prompt = build_prompt(title=title, artist=artist)
    # здесь ты вызываешь свою audio+LLM модель,
    # передавая audio_path и prompt, и возвращаешь строку-описание
    description = call_model(audio_path, prompt)
    return description

descriptions = []
for _, row in df.iterrows():
    audio_path = row["audio"]    # если там путь; если id — преобразуй
    artist = row["artist"]
    title = row["title"]
    desc = describe_track(audio_path, artist, title)
    descriptions.append(desc)

df["description"] = descriptions
df.to_parquet("data/processed/data_with_descriptions.parquet")